### PROYECTO 2


In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from scipy import stats
from scipy import optimize as opt  # para estimación por máxima verosimilitud
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.distributions.empirical_distribution import ECDF

import statsmodels.api as sm
from statsmodels.nonparametric.kde import KDEUnivariate

from statsmodels.nonparametric.kernel_density import KDEMultivariate
import matplotlib.tri as tri

In [ ]:
import os
os.getcwd()

In [ ]:
nom_fi_datos_Irrad = 'G11A_DATOS_IRRAD.csv'
nom_fi_datos_Util = 'G11A_DATOS_UTIL.csv'

df_orig_Irrad = pd.read_csv(nom_fi_datos_Irrad)
df_orig_Util = pd.read_csv(nom_fi_datos_Util)

# parseamos la fecha (cadena) para que sea un datetime con formato yyyy/mm/dd 
df_orig_Irrad['FECHA'] = pd.to_datetime(df_orig_Irrad['FECHA'], format='%Y-%m-%d')

s = df_orig_Irrad.loc[:,'FECHA']
df_orig_Irrad['FECHA'] =  s.dt.date

print('Tamaño de df_orig_Irrad con los datos cargados:', df_orig_Irrad.shape)
print('Dataframe con todos los datos leidos (filas: días, columnas: variables):')
print(df_orig_Irrad)

In [ ]:
# creamos nuevo df con los datos del dia y las horas a considerar
df_IRRADH06 = df_orig_Irrad.loc[:, ['MES', 'IRRADH06']]
df_IRRADH18 = df_orig_Irrad.loc[:, ['MES', 'IRRADH18']]

filtro= df_IRRADH06.MES==8
filtro_2= df_IRRADH18.MES==8

df_IRRADH06_AGOSTO = df_orig_Irrad.loc[filtro, ['MES', 'IRRADH06']]
df_IRRADH18_AGOSTO =df_orig_Irrad.loc[filtro_2, ['MES', 'IRRADH18']]

df_IRRADH18_AGOSTO

In [ ]:
#Necesito realizar una distribucion de las medias. Utilizo una t de student ya que no conozco la varianza

n=186
#calculo una media
media_IRRAD06=np.mean(df_IRRADH06_AGOSTO.IRRADH06)
media_IRRAD18=np.mean(df_IRRADH18_AGOSTO.IRRADH18)

#calculo uns std
sx_IRRAD06=np.std(df_IRRADH06_AGOSTO.IRRADH06, ddof=1)
sx_IRRAD18=np.std(df_IRRADH18_AGOSTO.IRRADH18, ddof=1)


#Utilizo t'student

IRRADH06_media=stats.t(n-1,loc=media_IRRAD06, scale=sx_IRRAD06/np.sqrt(n))
IRRADH18_media=stats.t(n-1,loc=media_IRRAD18, scale=sx_IRRAD18/np.sqrt(n))

#Calculo los intervalos de confianza

gama=0.9
alfa=1-gama

#Calculo limites

lim_inf_IRRADH06=IRRADH06_media.ppf(alfa/2)
lim_inf_IRRADH18=IRRADH18_media.ppf(alfa/2)

lim_sup_IRRADH06=IRRADH06_media.ppf(1-alfa/2)
lim_sup_IRRADH18=IRRADH18_media.ppf(1-alfa/2)

print(lim_inf_IRRADH06,lim_sup_IRRADH06)



In [ ]:

# puntos para representar la distribución y el intervalo
xs = np.linspace(IRRADH06_media.ppf(0.00001), IRRADH06_media.ppf(1-0.00001), 1000)
xIC = np.linspace(lim_inf_IRRADH06, lim_sup_IRRADH06, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, IRRADH06_media.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
plt.fill_between( xIC, IRRADH06_media.pdf(xIC), color="green", alpha=0.1, label=f'Probabilidad asociada ($\gamma=${gama})')
plt.vlines(media_IRRAD06, 0, np.max(IRRADH06_media.pdf(xs)), color='black', label=f'Estimación puntual media {media_IRRAD06}')
plt.errorbar(media_IRRAD06, -0.000001, xerr = np.array([[media_IRRAD06 - lim_inf_IRRADH06], [lim_sup_IRRADH06 - media_IRRAD06]]), fmt = 'o', label = f'IC [{lim_inf_IRRADH06}, {lim_sup_IRRADH06}] al {gama*100}%')
plt.title('Distribución del estimador media muestral de X e IC')
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
xs = np.linspace(IRRADH18_media.ppf(0.00001), IRRADH18_media.ppf(1-0.00001), 1000)
xIC = np.linspace(lim_inf_IRRADH18, lim_sup_IRRADH18, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, IRRADH18_media.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
plt.fill_between( xIC, IRRADH18_media.pdf(xIC), color="green", alpha=0.1, label=f'Probabilidad asociada ($\gamma=${gama})')
plt.vlines(media_IRRAD18, 0, np.max(IRRADH18_media.pdf(xs)), color='black', label=f'Estimación puntual media {media_IRRAD18}')
plt.errorbar(media_IRRAD18, -0.000001, xerr = np.array([[media_IRRAD18 - lim_inf_IRRADH18], [lim_sup_IRRADH18 - media_IRRAD18]]), fmt = 'o', label = f'IC [{lim_inf_IRRADH18}, {lim_sup_IRRADH18}] al {gama*100}%')
plt.title('Distribución del estimador media muestral de X e IC')
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
#CALCULO EL ESTADISTICO DE CONTRASTE 
#COMO EL CONTRASTE LO QUIERO HACER CON UNA MEDIA NULA, MU_0=0
print("H0= La media es 0")
print("H1=  La media es distinta de 0")

t0_IRRADH06 = media_IRRAD06/(sx_IRRAD06/np.sqrt(n))
t0_IRRADH18 = media_IRRAD18/(sx_IRRAD18/np.sqrt(n))


#PARA IRRADH06
df = n -1 # grados de libertad
T0_06 = stats.t(df)

# calculamos el pvalor (caso H1 BILATERAL)
pvalor_IRRADH06 = 2 * (1 - T0_06.cdf(abs(t0_IRRADH06)))

print(f'El valor observado del estadistico de contraste es {t0_IRRADH06}, con un p-valor asociado igual a {pvalor_IRRADH06}.')


# razonando con valores críticos
# nivel de significación
alfa = 0.1

# determina los valores críticos que definen la región de rechazo (bilateral)
t_crit_iz06 = T0_06.ppf(alfa/2)
t_crit_de06 = T0_06.ppf(1-alfa/2)


#PARA IRRADH18
df = n -1 # grados de libertad
T0_18 = stats.t(df)

# calculamos el pvalor (caso H1 BILATERAL)
pvalor_IRRADH18 = 2 * (1 - T0_18.cdf(abs(t0_IRRADH18)))

print(f'El valor observado del estadistico de contraste es {t0_IRRADH18}, con un p-valor asociado igual a {pvalor_IRRADH18}.')


# razonando con valores críticos
# nivel de significación
alfa = 0.1

# determina los valores críticos que definen la región de rechazo (bilateral)
t_crit_iz18 = T0_18.ppf(alfa/2)
t_crit_de18 = T0_18.ppf(1-alfa/2)

In [ ]:
#REPRESENTACION IRRADH06
# puntos para representar la distribución y las regiones de aceptación y rechazo
xmin = T0_06.ppf(0.000001)
xmax = T0_06.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRA = np.linspace(t_crit_iz06, xmax, 1000)
xRRiz = np.linspace(xmin, t_crit_iz06, 1000)
xRRde = np.linspace(t_crit_de06, xmax, 1000)

plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0_06.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')

plt.fill_between( xRA, T0_06.pdf(xRA), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRiz, T0_06.pdf(xRRiz), color="red", alpha=0.6, label=f'P(T0<{t_crit_iz06}) = {alfa/2})')
plt.fill_between( xRRde, T0_06.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de06}) = {alfa/2})')

plt.vlines(t0_IRRADH06, 0, np.max(T0_06.pdf(xs)), color='black', label=f't0 = {t0_IRRADH06}')

plt.hlines(-0.02, t_crit_iz06, xmax, color='green', label = f'RA: [{t_crit_iz06},{t_crit_de06}] al {alfa*100}%')
plt.hlines(-0.02, xmin, t_crit_iz06, color='red', linewidth = 6, label = f'RR (izq)')
plt.hlines(-0.02, t_crit_de06, xmax, color='red', linewidth = 6, label = f'RR (der)')

plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()


In [ ]:
#REPRESENTACION IRRADH18
# puntos para representar la distribución y las regiones de aceptación y rechazo
xmin = T0_18.ppf(0.000001)
xmax = T0_18.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRA = np.linspace(t_crit_iz18, xmax, 1000)
xRRiz = np.linspace(xmin, t_crit_iz18, 1000)
xRRde = np.linspace(t_crit_de18, xmax, 1000)

plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0_18.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')

plt.fill_between( xRA, T0_18.pdf(xRA), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRiz, T0_18.pdf(xRRiz), color="red", alpha=0.6, label=f'P(T0<{t_crit_iz18}) = {alfa/2})')
plt.fill_between( xRRde, T0_18.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de18}) = {alfa/2})')

plt.vlines(t0_IRRADH18, 0, np.max(T0_06.pdf(xs)), color='black', label=f't0 = {t0_IRRADH18}')

plt.hlines(-0.02, t_crit_iz18, xmax, color='green', label = f'RA: [{t_crit_iz18},{t_crit_de18}] al {alfa*100}%')
plt.hlines(-0.02, xmin, t_crit_iz18, color='red', linewidth = 6, label = f'RR (izq)')
plt.hlines(-0.02, t_crit_de18, xmax, color='red', linewidth = 6, label = f'RR (der)')

plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()


In [ ]:
###CASO UNILATERAl
print("H0= La media es enor 0")
print("H1=  La media es mayor que 0")


pvalor_IRRAD06=pvalor = 1 - T0_06.cdf(t0_IRRADH06)
pvalor_IRRAD18=pvalor = 1 - T0_18.cdf(t0_IRRADH18)

t_crit_uni06=T0_06.ppf(1-alfa)
t_crit_uni18=T0_18.ppf(1-alfa)


In [ ]:
### Representacion IRRADH06 unilateral

xmin = T0_06.ppf(0.000001)
xmax = T0_06.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRRde = np.linspace(t_crit_de06, xmax, 1000)
 
plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0_06.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')
 
plt.fill_between( xs, T0_06.pdf(xs), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRde, T0_06.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de06}) = {alfa}')
 
plt.vlines(t0_IRRADH06, 0, np.max(T0_06.pdf(xs)), color='black', label=f't0 = {t0_IRRADH06}')
 
plt.hlines(-0.02, xmin, t_crit_de06, color='green', label = f'RA: [{t_crit_iz06},{t_crit_de06}] al {alfa*100}%')
plt.hlines(-0.02, t_crit_de06, xmax, color='red', linewidth = 6, label = f'RR (der)')
 
plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()


In [ ]:
xmin = T0_06.ppf(0.000001)
xmax = T0_06.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRRde = np.linspace(t_crit_iz06, xmin, 1000)
 
plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0_06.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')
 
plt.fill_between( xs, T0_06.pdf(xs), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRde, T0_06.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de06}) = {alfa}')
 
plt.vlines(t0_IRRADH06, 0, np.max(T0_06.pdf(xs)), color='black', label=f't0 = {t0_IRRADH06}')
 
plt.hlines(-0.02, xmin, t_crit_de06, color='green', label = f'RA: [{t_crit_iz06},{t_crit_de06}] al {alfa*100}%')
plt.hlines(-0.02, xmin, t_crit_iz06, color='red', linewidth = 6, label = f'RR (der)')
 
plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()

In [ ]:
###IRRADH18
xmin = T0_18.ppf(0.000001)
xmax = T0_18.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRRde = np.linspace(t_crit_de18, xmax, 1000)
 
plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0_18.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')
 
plt.fill_between( xs, T0_18.pdf(xs), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRde, T0_18.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de18}) = {alfa}')
 
plt.vlines(t0_IRRADH18, 0, np.max(T0_18.pdf(xs)), color='black', label=f't0 = {t0_IRRADH18}')
 
plt.hlines(-0.02, xmin, t_crit_de18, color='green', label = f'RA: [{t_crit_iz18},{t_crit_de18}] al {alfa*100}%')
plt.hlines(-0.02, t_crit_de18, xmax, color='red', linewidth = 6, label = f'RR (der)')
 
plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()

In [ ]:
xmin = T0_18.ppf(0.000001)
xmax = T0_18.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRRde = np.linspace(t_crit_iz18, xmin, 1000)
 
plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0_18.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')
 
plt.fill_between( xs, T0_18.pdf(xs), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRde, T0_18.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de18}) = {alfa}')
 
plt.vlines(t0_IRRADH18, 0, np.max(T0_18.pdf(xs)), color='black', label=f't0 = {t0_IRRADH18}')
 
plt.hlines(-0.02, xmin, t_crit_de18, color='green', label = f'RA: [{t_crit_iz18},{t_crit_de18}] al {alfa*100}%')
plt.hlines(-0.02, xmin, t_crit_iz18, color='red', linewidth = 6, label = f'RR (der)')
 
plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()

### APARTADO 2

In [ ]:
for i in range(0,10,3):
    df_mes = df_orig_Util.loc[:, ['MES',f'UTILH0{i}']]
    x=np.array(df_mes[f'UTILH0{i}'])
    # nivel de significación
    alfa = 0.02

    # realiza el test
    ks = sm.stats.diagnostic.kstest_normal(x, dist='norm', pvalmethod='table')

    max_d   = ks[0]
    p_valor = ks[1]
    print(f'Contraste KS es una normal:: máxima distancia observada = {max_d} -> p_valor = {p_valor}')

    # concluye
    print(f'Como el pvalor < {alfa}, se puede rechazar H0 con un nivel de significación del {alfa*100}% ya que la distancia máxima observada es suficientemente grande.')
    print(f'Según el test no paramétrico KS se puede afirmar, con un nivel de significación del {alfa*100}%, que los datos de la muestra no siguen una distribución normal.')
    # nombre de la distribución teórica que se contrasta (norm, uniform, triang, expon, ...)
    distribucion_teo = stats.norm

    # genera qqplot de la variable x
    sm.qqplot(x, line='45', fit=True, dist=distribucion_teo)
    plt.title('')
    plt.grid(); plt.show()

In [ ]:
for i in range(12,22,3):
    df_mes = df_orig_Util.loc[:, ['MES',f'UTILH{i}']]
    x=np.array(df_mes[f'UTILH{i}'])
    # nivel de significación
    alfa = 0.02

    # realiza el test
    ks = sm.stats.diagnostic.kstest_normal(x, dist='norm', pvalmethod='table')

    max_d   = ks[0]
    p_valor = ks[1]
    print(f'Contraste KS es una normal:: máxima distancia observada = {max_d} -> p_valor = {p_valor}')

    # concluye
    print(f'Como el pvalor < {alfa}, se puede rechazar H0 con un nivel de significación del {alfa*100}% ya que la distancia máxima observada es suficientemente grande.')
    print(f'Según el test no paramétrico KS se puede afirmar, con un nivel de significación del {alfa*100}%, que los datos de la muestra no siguen una distribución normal.')
    # nombre de la distribución teórica que se contrasta (norm, uniform, triang, expon, ...)
    distribucion_teo = stats.norm

    # genera qqplot de la variable x
    sm.qqplot(x, line='45', fit=True, dist=distribucion_teo)
    plt.title('H0: Se distribuye como una normal')
    plt.grid(); plt.show()

In [ ]:
for i in range(6,10,3):
    
    df_mes = df_orig_Util.loc[:, ['MES',f'UTILH0{i}']]
    x=np.array(df_mes[f'UTILH0{i}'])
    n = len(x)
    dist = ['uniform','expon', 'norm', 'triang', 'beta']

    xs = np.linspace(min(x)-np.std(x),max(x)+np.std(x), 5000)

    plt.figure(1, figsize=(15, 6))
    plt.hist(x, bins=40, density = True, alpha = 0.4)

    # crea contendor con los resultados para elegir al final
    res_aju_distr = []

    for dist_nom in dist:
        dist1 = getattr(stats, dist_nom)
        params = dist1.fit(x)
        arg = params[:-2]
        loc = params[-2]
        scale = params[-1]

        if arg:
            pdf_fitted = dist1.pdf(xs, *arg, loc=loc, scale=scale)
            logLikelihood_med = np.sum(dist1.logpdf(x, *arg, loc=loc, scale=scale)) / n
        else:
            pdf_fitted = dist1.pdf(xs, loc=loc, scale=scale)
            logLikelihood_med = np.sum(dist1.logpdf(x, loc=loc, scale=scale)) / n

        # guarda resultado para mostrarlo al final
        res_aju_distr.append((dist_nom, logLikelihood_med, params))

        print(f'{dist_nom} :: \t logLikelihood_med = {logLikelihood_med} \t params:{params}')
        plt.plot(xs, pdf_fitted, linewidth = 3, label=f'logLikelihood_med({dist_nom}) = {logLikelihood_med}' )

    plt.legend(loc='upper left')
    plt.show()


    # tabla de resultados, ordenada de peor a mejor según logLikelihood_med
    res_aju_distr_ord = sorted(res_aju_distr, key=lambda x:x[1])#0,1,2 para nombre, maxima vero,params
    res_df = pd.DataFrame(res_aju_distr_ord, columns = ['Distribución', 'Media verosimilitud log', 'Parámetros'])
    res_df
    # estima el KDE

    # fija la función kernel a utilizar
    # “gau” for Gaussian,  “epa” for Epanechnikov, “uni” for uniform
    # “tri” for triangular, “cos” for cosine,  “biw” for biweight,  “triw” for triweight
    kernel_fun = 'gau'

    # fija el bandwidth (si auto lo estima la función de ajuste)
    tipo_bw = 'auto' # 'manual'

    if tipo_bw == 'auto':
        bandwidth = 'normal_reference' # otras opciones: bandwidth = 'scott', bandwidth = 'silverman'
    else:
        bandwidth = 0.6 # fija bandwidth exacto
    # estima kde
    kde = KDEUnivariate(x)
    kde.fit(kernel = kernel_fun, bw = bandwidth, fft = True)

    # calcula verosimilitud log media de ambas aproximaciones
    logLikelihood_med_kde = np.sum(np.log(kde.evaluate(x))) / n

    print(f'Verosimilitud logarítmica media de KDE: {logLikelihood_med_kde}')

    # fija nbins para mostrar histograma
    nbins_hist = 40

    plt.figure(1, figsize=(15, 5))
    plt.hist(x, nbins_hist, density = True, alpha = 0.4, label =f'Histograma nbins={nbins_hist}')
    plt.plot(kde.support, kde.density, color = 'black', linewidth = 3, label =f'pdf KDE bw={kde.bw}')

    # marca el máximo
    index_max_pdf = np.argmax(kde.density)   # <--- encuentra el argumento que hace máximo la función
    x_max_pdf = kde.support[index_max_pdf]
    max_pdf = kde.density[index_max_pdf]
    plt.plot(x_max_pdf, max_pdf, 's', color = 'green', markersize = 10, label =f'max pdf KDE ({x_max_pdf}, {max_pdf})')

    sns.rugplot(x, height = 0.025, lw = 0.4, color = 'red')
    plt.title(f'KDE con bandwidth = {kde.bw} | bw fijado con método: {kde.bw_method}')
    plt.grid(); plt.legend()
    plt.show()
    
for i in range(12,16,3):
    print("*"*100)
    df_mes = df_orig_Util.loc[:, ['MES',f'UTILH{i}']]
    x=np.array(df_mes[f'UTILH{i}'])
    n = len(x)
    # Indica la lista de distribuciones continuas que se prueban (ojo, puede tardar)
    dist = ['uniform','expon', 'norm', 'triang', 'beta']

    xs = np.linspace(min(x)-np.std(x),max(x)+np.std(x), 5000)

    plt.figure(1, figsize=(15, 6))
    plt.hist(x, bins=40, density = True, alpha = 0.4)

    # crea contendor con los resultados para elegir al final
    res_aju_distr = []

    for dist_nom in dist:
        dist1 = getattr(stats, dist_nom)
        params = dist1.fit(x)
        arg = params[:-2]
        loc = params[-2]
        scale = params[-1]

        if arg:
            pdf_fitted = dist1.pdf(xs, *arg, loc=loc, scale=scale)
            logLikelihood_med = np.sum(dist1.logpdf(x, *arg, loc=loc, scale=scale)) / n
        else:
            pdf_fitted = dist1.pdf(xs, loc=loc, scale=scale)
            logLikelihood_med = np.sum(dist1.logpdf(x, loc=loc, scale=scale)) / n

        # guarda resultado para mostrarlo al final
        res_aju_distr.append((dist_nom, logLikelihood_med, params))

        print(f'{dist_nom} :: \t logLikelihood_med = {logLikelihood_med} \t params:{params}')
        plt.plot(xs, pdf_fitted, linewidth = 3, label=f'logLikelihood_med({dist_nom}) = {logLikelihood_med}' )

    plt.legend(loc='upper left')
    plt.show()


    # tabla de resultados, ordenada de peor a mejor según logLikelihood_med
    res_aju_distr_ord = sorted(res_aju_distr, key=lambda x:x[1])#0,1,2 para nombre, maxima vero,params
    res_df = pd.DataFrame(res_aju_distr_ord, columns = ['Distribución', 'Media verosimilitud log', 'Parámetros'])
    res_df
    # estima el KDE

    # fija la función kernel a utilizar
    # “gau” for Gaussian,  “epa” for Epanechnikov, “uni” for uniform
    # “tri” for triangular, “cos” for cosine,  “biw” for biweight,  “triw” for triweight
    kernel_fun = 'gau'

    # fija el bandwidth (si auto lo estima la función de ajuste)
    tipo_bw = 'auto' # 'manual'

    if tipo_bw == 'auto':
        bandwidth = 'normal_reference' # otras opciones: bandwidth = 'scott', bandwidth = 'silverman'
    else:
        bandwidth = 0.6 # fija bandwidth exacto


    # estima kde
    kde = KDEUnivariate(x)
    kde.fit(kernel = kernel_fun, bw = bandwidth, fft = True)

    # calcula verosimilitud log media de ambas aproximaciones
    logLikelihood_med_kde = np.sum(np.log(kde.evaluate(x))) / n

    print(f'Verosimilitud logarítmica media de KDE: {logLikelihood_med_kde}')

    # fija nbins para mostrar histograma
    nbins_hist = 40

    plt.figure(1, figsize=(15, 5))
    plt.hist(x, nbins_hist, density = True, alpha = 0.4, label =f'Histograma nbins={nbins_hist}')
    plt.plot(kde.support, kde.density, color = 'black', linewidth = 3, label =f'pdf KDE bw={kde.bw}')

    # marca el máximo
    index_max_pdf = np.argmax(kde.density)   # <--- encuentra el argumento que hace máximo la función
    x_max_pdf = kde.support[index_max_pdf]
    max_pdf = kde.density[index_max_pdf]
    plt.plot(x_max_pdf, max_pdf, 's', color = 'green', markersize = 10, label =f'max pdf KDE ({x_max_pdf}, {max_pdf})')

    sns.rugplot(x, height = 0.025, lw = 0.4, color = 'red')
    plt.title(f'KDE con bandwidth = {kde.bw} | bw fijado con método: {kde.bw_method}')
    plt.grid(); plt.legend()
    plt.show()

### APARTADO 3

In [ ]:
df_orig_Util = pd.read_csv(nom_fi_datos_Util)

# parseamos la fecha (cadena) para que sea un datetime con formato yyyy/mm/dd 
df_orig_Util['FECHA'] = pd.to_datetime(df_orig_Util['FECHA'], format='%Y-%m-%d')

s = df_orig_Util.loc[:,'FECHA']
df_orig_Util['FECHA'] =  s.dt.date

print('Tamaño de df_orig_Util con los datos cargados:', df_orig_Util.shape)
print('Dataframe con todos los datos leidos (filas: días, columnas: variables):')
print(df_orig_Util)

In [ ]:
df1=df_orig_Util.UTILH06
df2=df_orig_Util.UTILH09
df3=df_orig_Util.UTILH12
df4=df_orig_Util.UTILH15
NIVELES_CONFIANZA=[0.8,0.95]
horas=[df1,df2,df3,df4]
for nivel_confianza in NIVELES_CONFIANZA:
    n=6
    for hora in horas:
        x=hora
        muestra_orig = (x,)  # tiene que ser una secuencia
        R = 9999
        estadistico = np.mean

        res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza,  n_resamples = R, method = 'BCA', random_state = 999)

        # devuelve como resultado el IC y el SE del estadístico
        lim_inf, lim_sup = res_BCa.confidence_interval
        SE_BCa = res_BCa.standard_error

        print(f'IC {100*nivel_confianza}% para la hora {n} , estimado según método BCa para la media: [{lim_inf}, {lim_sup}]')
        print(f'Error estándar del estimador media muestral, estimado según método BCa: {SE_BCa}\n')
        n+=3
        Xmedia = stats.t(n-1, loc=np.mean(x), scale = SE_BCa)
        x_media = np.mean(x)
        xs = np.linspace(Xmedia.ppf(0.00001), Xmedia.ppf(1-0.00001), 1000)
        xIC = np.linspace(lim_inf, lim_sup, 1000)
        
        plt.figure(1, figsize=(10, 5))
        plt.plot(xs, Xmedia.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
        plt.fill_between( xIC, Xmedia.pdf(xIC), color="green", alpha=0.3, label=f'Probabilidad asociada ($\gamma=${nivel_confianza})')
        plt.vlines(x_media, 0, np.max(Xmedia.pdf(xs)), color='black', label=f'Estimación puntual media {x_media}')
        plt.errorbar(x_media, -0.01, xerr = np.array([[x_media - lim_inf], [lim_sup - x_media]]), fmt = 'o', label = f'IC [{lim_inf}, {lim_sup}] al {nivel_confianza*100}%')
        plt.title('Distribución del estimador media muestral de X e IC')
        plt.tight_layout()
        plt.legend()
        plt.show()





In [ ]:
print("Se observa como los pares de horas 06, 15 y 09, 12 tienen intervalos de confianza muy parecidos, casi identicos,habiendo tambien una minima diferencia en sus errores estandar")

In [ ]:
print("Al no variar dichos intervalos en funcion del nivel de confianza sabemos al 100% que el parametro estimado esta dentro de dicho intervalo")

### APARTADO 4

In [ ]:
#parametros de forma.
# crea la v.a. "VA" que sigue una Uniforme
a6= 0.003608
b6= 0.4748

inicio = a6
incremento = b6 - a6
VA1 = stats.uniform(loc=inicio, scale=incremento)

# genera n números aleatorios segun VA
tamaños = [10,100,1000]
for n in tamaños:
    x = VA1.rvs(size = n)
    fig = plt.figure(1, figsize=(15, 4))
    ax1=plt.subplot(1,3,1)
    plt.plot( x, 'o')
    plt.ylabel('x')
    plt.title(f'{n} números aleatorios segun U({a6},{b6}) en 06')

    plt.subplot(1,3,2, sharey = ax1)
    plt.hist(x, orientation = 'horizontal')
    plt.ylabel('x')
    plt.title(f'Histograma para {n} numeros aleatorios en 06')
    xs = np.linspace(VA1.ppf(0.01)-1, VA1.ppf(0.99)+1, 1000)
    plt.subplot(1,3,3, sharey = ax1)
    plt.plot(VA1.pdf(xs), xs, color="black")
    plt.ylabel('x')
    plt.title(f'Distribución teórica para {n} numeros aleatorios en 06')
    plt.show()
print('\n'*2)
loc = 0.0436
escalado = 0.513
VA2 = stats.expon(loc = loc, scale = escalado)
for n in tamaños:
    x = VA2.rvs(size = n)
    fig = plt.figure(1, figsize=(15, 4))
    ax1=plt.subplot(1,3,1)
    plt.plot( x, 'o')
    plt.ylabel('x')
    plt.title(f'{n} números aleatorios segun Ex({loc},{escalado}) en 09')

    plt.subplot(1,3,2, sharey = ax1)
    plt.hist(x, orientation = 'horizontal')
    plt.ylabel('x')
    plt.title(f'Histograma para {n} numeros aleatorios en 09')
    xs = np.linspace(VA2.ppf(0.01)-1, VA2.ppf(0.99)+1, 1000)
    plt.subplot(1,3,3, sharey = ax1)
    plt.plot(VA2.pdf(xs), xs, color="black")
    plt.ylabel('x')
    plt.title(f'Distribución teórica para {n} numeros aleatorios en 09')
    plt.show()
print('\n'*2)

a = 0.04 # min
c = 0.824 # moda 
b = 0.839 # max
loc = a
escalado = b-a
forma = (c-a)/escalado
VA3 = stats.triang(loc = loc, scale = escalado, c = forma)
x = VA3.rvs(size = n)
for n in tamaños:
    x = VA3.rvs(size = n)
    fig = plt.figure(1, figsize=(15, 4))
    ax1=plt.subplot(1,3,1)
    plt.plot( x, 'o')
    plt.ylabel('x')
    plt.title(f'{n} números aleatorios segun Triang({a},{b},{c}) en 012')

    plt.subplot(1,3,2, sharey = ax1)
    plt.hist(x, orientation = 'horizontal')
    plt.ylabel('x')
    plt.title(f'Histograma para {n} numeros aleatorios en 012')
    xs = np.linspace(VA3.ppf(0.01)-1, VA3.ppf(0.99)+1, 1000)
    plt.subplot(1,3,3, sharey = ax1)
    plt.plot(VA3.pdf(xs), xs, color="black")
    plt.ylabel('x')
    plt.title(f'Distribución teórica para {n} numeros aleatorios en 012')
    plt.show()
print('\n'*2)
a= 0.0042
b= 0.517

inicio = a
incremento = b - a
VA4 = stats.uniform(loc=inicio, scale=incremento)
for n in tamaños:
    x = VA4.rvs(size = n)
    fig = plt.figure(1, figsize=(15, 4))
    ax1=plt.subplot(1,3,1)
    plt.plot( x, 'o')
    plt.ylabel('x')
    plt.title(f'{n} números aleatorios segun U({a},{b}) en 015')

    plt.subplot(1,3,2, sharey = ax1)
    plt.hist(x, orientation = 'horizontal')
    plt.ylabel('x')
    plt.title(f'Histograma para {n} numeros aleatorios en 015')
    xs = np.linspace(VA4.ppf(0.01)-1, VA4.ppf(0.99)+1, 1000)
    plt.subplot(1,3,3, sharey = ax1)
    plt.plot(VA4.pdf(xs), xs, color="black")
    plt.ylabel('x')
    plt.title(f'Distribución teórica para {n} numeros aleatorios en 0')
    plt.show()

In [ ]:
for n in tamaños:
    np.random.seed(99)
    x1 = np.random.uniform(a6,b6, n)
    distribucion_teo = stats.beta
    sm.qqplot(x1, line='45', fit=True, dist=distribucion_teo)
    plt.title(f'H0:En la hora 06 Se distribuye como una Uniforme con {n} muestras')
    plt.grid(); plt.show()
for n in tamaños:
    np.random.seed(99)
    x1 = np.random.exponential(scale = escalado9,size = n) + loc9
    distribucion_teo = stats.beta
    sm.qqplot(x1, line='45', fit=True, dist=distribucion_teo)
    plt.title(f'H0:En la hora 09 Se distribuye como una Exponencial con {n} muestras')
    plt.grid(); plt.show()
for n in tamaños:
    np.random.seed(99)
    x1 = np.random.triangular(loc12,escalado12,forma12, n)
    distribucion_teo = stats.beta
    sm.qqplot(x1, line='45', fit=True, dist=distribucion_teo)
    plt.title(f'H0:En la hora 012 Se distribuye como una Triangular con {n} muestras')
    plt.grid(); plt.show()
for n in tamaños:
    np.random.seed(99)
    x1 = np.random.uniform(0.004,0.52, n)
    distribucion_teo = stats.beta
    sm.qqplot(x1, line='45', fit=True, dist=distribucion_teo)
    plt.title(f'H0:En la hora 015 Se distribuye como una Uniforme con {n} muestras')
    plt.grid(); plt.show()